# Reward-model experiment — beat the 0.63 ceiling

Research-backed levers vs the old run: **(1)** train+eval on **cleaned** UltraFeedback (`argilla/...-cleaned`; the H4 binarization mislabels ~50% of pairs), **(2)** init the RM from **Qwen2.5-0.5B-Instruct** (not a weak 8k-SFT; +3–8 pts in controlled ablations), **(3)** bigger effective batch, 1 epoch, no contrast filter. Reports accuracy on the **cleaned** held-out set *and* the old H4 `test_prefs` (the 0.63 yardstick). GPU-adaptive (T4/P100). RM-only (~2.5–3.5 h).

In [ ]:
import torch, json
cap  = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100 = tuple(cap) == (6, 0)
json.dump({'p100': P100}, open('/tmp/gpu.json', 'w'))
print(f'GPU: {name}  cap={cap}  ->  ' + ('P100: torch 2.4.1 + fp32' if P100 else 'T4/base: bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'], check=True)
    print('installed torch 2.4.1 for the P100')

In [ ]:
import os
REPO_URL = 'https://github.com/TheYellowDuck/RLHF-pipeline.git'
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q $REPO_URL /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python scripts/smoke_test.py 2>&1 | tail -3

## Config

In [ ]:
import json
P100 = json.load(open('/tmp/gpu.json'))['p100']
MODEL      = 'Qwen/Qwen2.5-0.5B-Instruct'                              # strong instruct backbone (lever #2)
DATA_CLEAN = 'argilla/ultrafeedback-binarized-preferences-cleaned'     # re-binarized, clean labels (lever #1)
DATA_H4    = 'HuggingFaceH4/ultrafeedback_binarized'                   # old (buggy) set = 0.63 yardstick
RM_SAMPLES = 10000          # ~2.5h on T4 to fit a 4h quota window; cleaned+instruct are the levers
DTYPE, BF16 = ('float32', 'false') if P100 else ('bfloat16', 'true')
print(f'GPU={"P100/fp32" if P100 else "T4/bf16"}  model={MODEL}  data={DATA_CLEAN}  samples={RM_SAMPLES}')

## Train reward model  (Instruct init + cleaned data, 1 epoch, eff. batch 16)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE \
  -o data.name=$DATA_CLEAN -o data.train_split='train[2000:]' -o data.eval_split='train[:2000]' \
  -o data.max_samples=$RM_SAMPLES -o data.max_eval_samples=2000 -o data.max_length=512 \
  -o train.epochs=1 -o train.batch_size=4 -o train.grad_accum=4 -o train.bf16=$BF16 \
  -o train.lr=1.0e-5 -o train.label_smoothing=0.0 -o data.max_pair_similarity=1.0 \
  -o output_dir=checkpoints/reward_model

## Evaluate — cleaned held-out (the real number) + old H4 test (vs 0.63)

In [ ]:
import subprocess
def run(cmd):
    print('$', cmd, flush=True)
    o = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(o.stdout[-600:]);
    if o.returncode: print(o.stderr[-1200:])
    return o.stdout

clean = run("python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model "
            f"--data {DATA_CLEAN} --split 'train[:2000]' --max-samples 2000")
h4    = run("python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model "
            f"--data {DATA_H4} --split test_prefs --max-samples 2000")
md = (f'# Reward-model experiment results\n\n- backbone: `{MODEL}`\n- train data: `{DATA_CLEAN}` '
      f'({RM_SAMPLES} pairs, 1 epoch)\n\n'
      f'## Accuracy on CLEANED held-out (the meaningful number)\n```\n{clean}\n```\n\n'
      f'## Accuracy on old H4 test_prefs (vs prior 0.63 baseline)\n```\n{h4}\n```\n')
open('/kaggle/working/RESULTS.md','w').write(md)
print('\nSaved -> /kaggle/working/RESULTS.md')

### Read
If **cleaned** accuracy ≫ 0.63, the old ceiling was largely mislabeled data. Compare the H4 number to the historical 0.63 to see how much was label noise vs real capacity.